# Week 2 Day 3 Challenge - a handler can reject a payout

The challenge: add a way for the handler to reject a payout. The handler's answer carries `approved` and a `note`, and when a handler rejects a payout, the note goes back to the agent as a message so it can look at the claim again. Try it on CLM-1011, a lost suitcase where the payout is easy to get wrong. The handler paid $1,150.

The stretch challenge: on a second computer with the repo and the same `DATABASE_URL`, build the graph and approve a claim that was paused on the first one.

The graph is the lab's, with these changes:

| | Lab | Challenge |
|---|---|---|
| The handler's answer | `{"handler": ...}` | `{"handler": ..., "approved": ..., "note": ...}` |
| The approval node | Records who approved | On a rejection, adds the handler's note to the messages, with the payout it rejects |
| After the approval node | Always pay | Pay an approval, send a rejection back to the agent |
| After the decide node | A payout over $1,000 waits for a handler | The same, and a claim a handler rejected always goes back to a handler, whatever the new amount |
| A limit | | After two rejections the claim is referred, so the agent and the handler can't go round forever |

## Setup

The same imports, Supabase connection and payout tool as the lab. The lab's tables stay as they were: this notebook pays into its own `challenge_payments` table, and its threads are named `challenge-...`.

In [ ]:
import json
import operator
import os
import subprocess
import sys
from pathlib import Path
from typing import Annotated, Literal, TypedDict

import pandas as pd
import psycopg
from dotenv import load_dotenv
from IPython.display import Image, display
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command, interrupt
from psycopg.rows import dict_row
from pydantic import BaseModel, Field

# This notebook sits two folders below the repo's root, and the payout rules are in labs/
ROOT = Path("../..")
sys.path.append(str(ROOT / "labs"))
from week2day1_payout import Benefit, Line, Payout, work_out_payout  # noqa: E402

In [ ]:
load_dotenv(ROOT / ".env", override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
DATABASE_URL = os.getenv("DATABASE_URL")

connection = psycopg.connect(DATABASE_URL, autocommit=True, prepare_threshold=0, row_factory=dict_row, options="-c search_path=lab")
connection.execute("CREATE SCHEMA IF NOT EXISTS lab")
connection.execute("""
CREATE TABLE IF NOT EXISTS challenge_payments (
    claim_id TEXT PRIMARY KEY,
    amount NUMERIC NOT NULL,
    approved_by TEXT,
    paid_at TIMESTAMPTZ DEFAULT now()
)
""")

checkpointer = PostgresSaver(connection)
checkpointer.setup()
print(connection.execute("SELECT version()").fetchone()["version"])

In [ ]:
DATA = ROOT / "data"


def load(name):
    """Read one of UnderCover's JSON files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = {claim["claim_id"]: claim for claim in load("claims.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
APPROVAL_LIMIT = load("company.json")["payout_approval_limit"]
MAX_REJECTIONS = 2

INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use the payout tool to work out the payout on an approved claim. When you have decided, give your decision, the payout and your reasons.
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)


luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)
model_with_tools = luna.bind_tools([calculate_payout])
decider = luna.with_structured_output(ClaimDecision)
system_message = SystemMessage(f"{INSTRUCTIONS}\n{policy_wording}\n")

## The graph

The state has two new fields. `rejections` counts the handler's rejections. `notes` keeps the handler's notes, with `operator.add` as its reducer, so each note is added to the list.

In [ ]:
class ClaimState(TypedDict):
    """What the graph keeps for one claim."""

    messages: Annotated[list, add_messages]
    claim_id: str
    decision: str
    payout: float
    rationale: str
    approved_by: str
    rejections: int
    notes: Annotated[list, operator.add]

The agent and decide nodes are the lab's.

In [ ]:
def agent(state: ClaimState):
    """Call the model with the conversation so far."""
    return {"messages": [model_with_tools.invoke([system_message, *state["messages"]])]}


def decide(state: ClaimState):
    """Turn the agent's work into a decision."""
    result = decider.invoke([system_message, *state["messages"], HumanMessage("Record your decision on this claim.")])
    return {"decision": result.decision, "payout": result.payout, "rationale": result.rationale}

### The approval node

The handler's answer now says whether they approve. On a rejection the node adds the note to the messages. The decide node's answer isn't in the messages, so the note names the payout the handler rejected; otherwise the agent wouldn't know which figure was wrong.

At the second rejection the node refers the claim, with no payout and a rationale that says why. The rejected payouts stay in `notes`.

In [ ]:
def approval(state: ClaimState):
    """Wait for a handler to approve or reject the payout."""
    print(f"{state['claim_id']} needs a handler to approve ${state['payout']:,.2f}")
    answer = interrupt({"claim_id": state["claim_id"], "payout": state["payout"], "rationale": state["rationale"]})
    if answer["approved"]:
        return {"approved_by": answer["handler"]}
    note = f"{answer['handler']} rejected the payout of ${state['payout']:,.2f}: {answer['note']}"
    rejections = state.get("rejections", 0) + 1
    update = {"messages": [HumanMessage(note)], "rejections": rejections, "notes": [note]}
    if rejections >= MAX_REJECTIONS:
        update |= {"decision": "refer", "payout": 0.0, "rationale": f"Referred after {rejections} rejections by a handler"}
    return update

The pay node is the lab's, paying into this notebook's table.

In [ ]:
def pay(state: ClaimState):
    """Pay the claim, unless it has been paid already."""
    cursor = connection.execute(
        "INSERT INTO challenge_payments (claim_id, amount, approved_by) VALUES (%s, %s, %s) ON CONFLICT (claim_id) DO NOTHING",
        (state["claim_id"], state["payout"], state.get("approved_by")),
    )
    if cursor.rowcount:
        print(f"Paid {state['claim_id']} ${state['payout']:,.2f}")
    else:
        print(f"{state['claim_id']} was already paid")

### The edges

`after_decide` gains one rule: once a handler has rejected a payout, the claim goes back to a handler even if the new payout is under the limit. Otherwise a rejected $1,500 that the agent cut to $950 would be paid without anyone seeing it.

`after_approval` is new, in place of the lab's fixed edge from approval to pay.

In [ ]:
def after_agent(state: ClaimState):
    """Run the tools the model asked for, or move on to the decision."""
    if state["messages"][-1].tool_calls:
        return "tools"
    return "decide"


def after_decide(state: ClaimState):
    """Pay an approval, after a handler's approval if it's over the limit or a handler rejected it before."""
    if state["decision"] != "approve":
        return END
    if state["payout"] > APPROVAL_LIMIT or state.get("rejections"):
        return "approval"
    return "pay"


def after_approval(state: ClaimState):
    """Pay an approved payout, send a rejected one back to the agent, or stop at a referral."""
    if state.get("approved_by"):
        return "pay"
    if state["decision"] == "refer":
        return END
    return "agent"

In [ ]:
builder = StateGraph(ClaimState)
builder.add_node("agent", agent)
builder.add_node("tools", ToolNode([calculate_payout]))
builder.add_node("decide", decide)
builder.add_node("approval", approval)
builder.add_node("pay", pay)

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", after_agent, ["tools", "decide"])
builder.add_edge("tools", "agent")
builder.add_conditional_edges("decide", after_decide, ["approval", "pay", END])
builder.add_conditional_edges("approval", after_approval, ["pay", "agent", END])
builder.add_edge("pay", END)

graph = builder.compile(checkpointer=checkpointer)
display(Image(graph.get_graph().draw_mermaid_png()))

The approval node now has three ways out: pay, back to the agent, or the end.

## CLM-1011: reject, then approve

The best man's lost suitcase, on a premium plan: a $300 per-item limit, a $1,500 maximum and a $50 deductible, with $350 paid by the airline. The handler paid $1,150.

`start` deletes a thread and its payment, then runs the claim until it pauses or ends. `answer` resumes it with a handler's answer.

In [ ]:
def start(thread_id, claim_id):
    """Run a claim on a fresh thread until it pauses or ends."""
    checkpointer.delete_thread(thread_id)
    connection.execute("DELETE FROM challenge_payments WHERE claim_id = %s", (thread_id,))
    config = {"configurable": {"thread_id": thread_id}}
    claim = {"messages": [HumanMessage(make_user_prompt(claims[claim_id]))], "claim_id": thread_id, "rejections": 0}
    return graph.invoke(claim, config, durability="sync")


def answer(thread_id, handler, approved, note=""):
    """Resume a paused thread with a handler's answer."""
    config = {"configurable": {"thread_id": thread_id}}
    return graph.invoke(Command(resume={"handler": handler, "approved": approved, "note": note}), config, durability="sync")


def payments():
    """The payments this notebook made."""
    return pd.DataFrame(connection.execute("SELECT * FROM challenge_payments ORDER BY paid_at").fetchall())

The claim id in the state is the thread id, so each run of the same claim gets its own payment row.

The handler's note points to the rule in section 5 that the agent should look at again, without giving the amount. I wrote it from the policy wording, not from the answer key.

`handle` plays the handler on one run of CLM-1011. It rejects the first payout with a note, unless the payout is already the handler's $1,150 and `reject_right_payouts` is off, and then it approves whatever the agent proposes next.

In [ ]:
HANDLER_PAID = 1150
NOTE = (
    "The payout is too high. Section 5 says a pair or a set counts as one item, such as a pair of shoes, "
    "a suit or a camera with its lens. Look again at which of the items form a set before you apply the per-item limit."
)
VAGUE_NOTE = "Please check the payout again."


def handle(thread_id, note=NOTE, reject_right_payouts=False):
    """Run CLM-1011 with a handler who rejects the first payout once, then approves the next one."""
    first = start(thread_id, "CLM-1011")
    row = {"thread": thread_id, "first decision": first["decision"], "first payout": first["payout"]}
    if "__interrupt__" in first:
        if first["payout"] == HANDLER_PAID and not reject_right_payouts:
            answer(thread_id, "Dana Whitfield", approved=True)
        else:
            second = answer(thread_id, "Dana Whitfield", approved=False, note=note)
            row |= {"after the note": second["decision"], "second payout": second["payout"]}
            if "__interrupt__" in second:
                answer(thread_id, "Dana Whitfield", approved=True)
    paid = connection.execute("SELECT amount FROM challenge_payments WHERE claim_id = %s", (thread_id,)).fetchone()
    row["paid"] = float(paid["amount"]) if paid else None
    row["right"] = row["paid"] == HANDLER_PAID
    return row

## Five runs: the handler rejects a wrong payout

The model's answers move between runs, so five runs. The handler approves a first payout of $1,150 straight away and rejects any other with the note.

In [ ]:
wrong_rows = [handle(f"challenge-CLM-1011-run{run}") for run in range(1, 6)]
pd.DataFrame(wrong_rows)

## Inside a rejected run

The first run the handler rejected: the note, the lines the agent sent to the payout tool before and after it, and the thread's checkpoints, newest first. The checkpoints show the way back from the approval node to the agent.

In [ ]:
def show_thread(thread_id):
    """A thread's notes, its calls to the payout tool and its checkpoints."""
    config = {"configurable": {"thread_id": thread_id}}
    for note in graph.get_state(config).values["notes"]:
        print("Note:", note)
    for message in graph.get_state(config).values["messages"]:
        for call in getattr(message, "tool_calls", None) or []:
            lines = [f"{line['description']} ${line['amount']:,.0f}" + (f" x{line['quantity']}" if line.get("quantity", 1) > 1 else "") for line in call["args"]["lines"]]
            print("Payout tool:", "; ".join(lines))
    for snapshot in graph.get_state_history(config):
        print(snapshot.metadata["step"], snapshot.next)


rejected = next((row["thread"] for row in wrong_rows if "second payout" in row), None)
if rejected:
    show_thread(rejected)
else:
    print("The handler didn't reject any of the five runs")

## Rejecting a right payout

A handler can be wrong too. Here the handler rejects the first payout whatever it is, with a note that gives no reason. An agent that only wants to please would change a right payout; one that checks its work would keep it.

In [ ]:
vague_rows = [handle(f"challenge-CLM-1011-vague{run}", note=VAGUE_NOTE, reject_right_payouts=True) for run in range(1, 4)]
pd.DataFrame(vague_rows)

## Two rejections

A handler who rejects twice. At the second rejection the approval node refers the claim, and the graph ends with nothing paid.

In [ ]:
thread_id = "challenge-CLM-1011-limit"
result = start(thread_id, "CLM-1011")
while "__interrupt__" in result:
    result = answer(thread_id, "Priya Raman", approved=False, note=VAGUE_NOTE)

state = graph.get_state({"configurable": {"thread_id": thread_id}})
print(state.next, state.values["decision"], state.values["rejections"], f"${state.values['payout']:,.2f}", state.values["rationale"])
for note in state.values["notes"]:
    print(note)
print("Paid:", connection.execute("SELECT count(*) FROM challenge_payments WHERE claim_id = %s", (thread_id,)).fetchone()["count"])

## Never pay twice

The pay node still checks the claim id, so calling it again on a paid thread does nothing.

In [ ]:
pay(graph.get_state({"configurable": {"thread_id": "challenge-CLM-1011-run1"}}).values)

## Stretch: a second computer

I have one computer, so the second computer is a second process. The notebook pauses the tornado claim, CLM-1008. Then `week2day3_second_computer.py`, in a process of its own, builds the same graph from the repo, finds the thread in Supabase by its id and approves it. It shares nothing with this notebook but `DATABASE_URL`: no memory, no connection, no graph.

On a real second computer the steps are the same: clone the repo, put the same `DATABASE_URL` and `OPENROUTER_API_KEY` in `.env`, `uv sync`, and run the script.

In [ ]:
result = start("challenge-CLM-1008", "CLM-1008")
print(f"Process {os.getpid()} paused CLM-1008 at {graph.get_state({'configurable': {'thread_id': 'challenge-CLM-1008'}}).next}: ${result['payout']:,.2f}")

In [ ]:
second_computer = subprocess.run(
    [sys.executable, "week2day3_second_computer.py", "challenge-CLM-1008", "Priya Raman"], capture_output=True, text=True, encoding="utf-8", check=True
)
print(second_computer.stdout, second_computer.stderr)

The payment the other process made, and the state this notebook now reads back from Supabase.

In [ ]:
print(graph.get_state({"configurable": {"thread_id": "challenge-CLM-1008"}}).next)
payments()

## Findings

- **The rejection loop works.** A rejected payout goes back to the agent with the handler's note, nothing is paid while the claim waits, and the agent's new payout waits for a handler again. The checkpoints of the rejected run show the way round: approval at step 4, then agent, tools, agent and decide, and approval again at step 9 before the payment.
- **With the payout tool, Luna often gets CLM-1011 right first time.** 3 of the 5 runs proposed $1,150 straight away. In my Week 1 Day 4 challenge, without the tool, the models got this claim wrong in nearly every try. The tool's description says that a pair or a set counts as one item.
- **A note that names the rule fixed both wrong payouts.** Run 3 counted the dinner jacket and the trousers as two items and proposed $1,330. Run 4 proposed $1,370, which is what counting the camera and its lens as two items gives. After the note, both proposed $1,150, so all five runs were paid right.
- **A rejection with no reason did harm.** With "Please check the payout again.", the agent changed a right $1,150 to $1,370 in one run. The two wrong first payouts stayed wrong: $1,370 became $1,300, and $1,500 stayed $1,500. None of the three was paid right. In the two-rejections run, the agent also changed a right $1,150, to $1,500. Both times the handler rejected a right payout without a reason, the agent changed it to a wrong one. Three runs is a small sample, but the lesson for the product is to require a reason when a handler rejects a payout.
- **The limit holds.** After two rejections the claim is referred with a payout of $0 and a rationale that says why, and nothing is paid. The rejected payouts stay in the notes.
- **Stretch.** A second process, with a different process id, found CLM-1008 waiting in Supabase, approved it and paid $2,240. This notebook then read the finished thread back. I ran it on one computer; the steps for a real second computer are in the stretch section.
- **The approval node prints twice on each resume.** Code before `interrupt()` runs again when the thread resumes, as the lab explains, so the payment stays in a node of its own.
- **Cost:** about a cent on GPT-6 Luna for the whole notebook.